## Pruebas del Algoritmo de Recomendación
Se cargan y preparan los datos, luego se prueba la función con los tres niveles.

In [ ]:
import pandas as pd
import numpy as np

def recommend_routes(level, df):
    # Filtrar rutas según el nivel ingresado
    if level == "Beginner":
        routes = df[df["difficulty_score"] <= 0.33]
    elif level == "Intermediate":
        routes = df[(df["difficulty_score"] > 0.33) & (df["difficulty_score"] <= 0.66)]
    elif level == "Advanced":
        routes = df[df["difficulty_score"] > 0.66]
    else:
        print("Nivel no válido. Usa: Beginner, Intermediate o Advanced")
        return None

    return routes.sort_values("difficulty_score")[
        ["name", "distance_km", "average_speed_kmh", "elevation_m", "speed_range", "difficulty_score"]
    ].head(10)

df = pd.read_csv('dataset/strava_data.csv')
df = df[df['sport_type'] == 'Ride']
df = df[["name", "distance", "moving_time", "total_elevation_gain", "average_speed", "sport_type", "date"]]
df = df.drop_duplicates()
df = df[df["average_speed"] > 5]
df = df[df["average_speed"] < 60]

### Clasificación de nivel por velocidad

In [ ]:
df["distance"] = df["distance"] * 1.60934
df["average_speed"] = df["average_speed"] * 1.60934
df["total_elevation_gain"] = df["total_elevation_gain"] * 0.3048
df["date"] = pd.to_datetime(df["date"])

df = df.rename(columns={
    "distance": "distance_km",
    "average_speed": "average_speed_kmh",
    "total_elevation_gain": "elevation_m"
})

df = df.assign(
    elev_norm=(df["elevation_m"] - df["elevation_m"].min()) / (df["elevation_m"].max() - df["elevation_m"].min()),
    dist_norm=(df["distance_km"] - df["distance_km"].min()) / (df["distance_km"].max() - df["distance_km"].min()),
    speed_norm=(df["average_speed_kmh"] - df["average_speed_kmh"].min()) / (df["average_speed_kmh"].max() - df["average_speed_kmh"].min())
)

df = df.assign(
    difficulty_score=(
        df["elev_norm"] * 0.5 +
        df["dist_norm"] * 0.3 +
        df["speed_norm"] * 0.2
    )
)

### Resultados por nivel

In [ ]:
print("=== Rutas para Beginner ===")
recommend_routes("Beginner", df)

In [ ]:
print("=== Rutas para Intermediate ===")
recommend_routes("Intermediate", df)

In [ ]:
print("=== Rutas para Advanced ===")
recommend_routes("Advanced", df)